# From Quarks to Quark-Gluon Plasma## Act 1: What's Inside a Proton?**Vanderbilt Programs for Talented Youth | Instructor: Jennifer James, Ph.D. Candidate**-Welcome! Over the next few hours you're going to analyze real and simulated particle physics data: the same kind of data that physicists at major laboratories around the world work with every day.We'll build up to one of the most dramatic discoveries in modern physics: a state of matter called the **Quark-Gluon Plasma (QGP)**: a soup of free quarks and gluons that filled the entire universe in the first microseconds after the Big Bang, and that we can recreate today by smashing gold nuclei together at nearly the speed of light.But first: what exactly is a proton made of? That's where we start.-### How to use this notebookThis document is a **Jupyter Notebook**. It contains two types of cells:- **Text cells** (like this one) contain explanations and instructions- **Code cells** (gray boxes) contain Python code that you can run**To run a code cell:** click on it, then press `Shift + Enter`.The output (numbers, plots, tables) will appear directly below the cell. Run the cells **in order from top to bottom**: each one builds on the one before it.Let's start with the classic first program in any language:

In [ ]:
print('Hello, particle physics!')

-## Part 1: Python BasicsPython on its own is useful, but physicists rely on powerful **libraries**: collections of pre-written tools that make complex tasks easy. We'll use three:| Library | What it does ||-|-|| `numpy` | Fast math on large arrays of numbers || `pandas` | Organizes data into tables (called DataFrames) || `matplotlib` | Makes plots and histograms |Run the cell below to load all three. Nothing will appear: that's normal. Loading a library is like turning on a tool before you use it.

In [ ]:
# The 'as' keyword gives each library a short nickname we'll use laterimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltprint('Libraries loaded successfully!')

### A quick look at PythonBefore we dive into physics, here are a few Python concepts you'll see throughout this notebook.

In [ ]:
# Variables store valuesproton_mass = 0.938 # in GeV/c^2electron_mass = 0.000511 # in GeV/c^2print('Proton mass:', proton_mass, 'GeV/c^2')print('Electron mass:', electron_mass, 'GeV/c^2')print('The proton is', round(proton_mass / electron_mass), 'times heavier than the electron')

In [ ]:
# Lists store many values togetherquark_names = ['up', 'down', 'strange', 'charm', 'bottom', 'top']quark_charges = [2/3, -1/3, -1/3, 2/3, -1/3, 2/3] # in units of electron charge# A proton is made of two up quarks and one down quarkproton_charge = 2 * (2/3) + 1 * (-1/3)print('Proton charge (should be +1):', round(proton_charge, 2))# A neutron is made of one up quark and two down quarksneutron_charge = 1 * (2/3) + 2 * (-1/3)print('Neutron charge (should be 0):', round(neutron_charge, 2))

-## Part 2: Setting Up a SimulationWe can't reach inside a proton with our hands. Instead, physicists use **particle accelerators** to fire high-energy particles at each other and study what comes out.One powerful technique is **electron-proton (e+p) scattering**: fire a high-energy electron at a proton, and the electron exchanges a virtual photon (γ*) with one of the quarks inside. The quark gets knocked out, and we study the debris to understand proton structure. This is called **Deep Inelastic Scattering (DIS)**.```electron ───────────────> ~~~γ*~~~> quark (knocked out!)|proton ──────────────────────────────> remnant fragments```Rather than running a real accelerator, we'll use **PYTHIA8**: a state-of-the-art Monte Carlo simulation program that physicists worldwide use to model particle collisions. It uses the known laws of quantum chromodynamics (QCD) to simulate what happens when particles collide.Run the cell below to load PYTHIA8 and set up our e+p collision:

In [ ]:
import pythia8# Create the PYTHIA simulation objectpythia = pythia8.Pythia()# Set up the beams: unequal energies (like the HERA accelerator in Germany)pythia.readString("Beams:frameType = 2")# Beam A: proton (particle ID 2212), 920 GeVpythia.readString("Beams:idA = 2212")pythia.settings.parm("Beams:eA", 920)# Beam B: electron (particle ID 11), 27.5 GeVpythia.readString("Beams:idB = 11")pythia.settings.parm("Beams:eB", 27.5)# Physics process: Deep Inelastic Scattering via virtual photon/Z exchangepythia.readString("WeakBosonExchange:ff2ff(t:gmZ) = on")pythia.settings.parm("PhaseSpace:Q2Min", 25) # minimum momentum transferpythia.readString("SpaceShower:dipoleRecoil = on")pythia.readString("SpaceShower:pTmaxMatch = 2")pythia.readString("PDF:lepton = off")pythia.readString("TimeShower:QEDshowerByL = off")# Initialize: PYTHIA prints a summary of the setuppythia.init()print('\nPYTHIA initialized! Ready to simulate collisions.')

-## Part 3: Your First CollisionNow let's simulate **one single collision** and look at every particle that comes out.Each particle is described by several quantities that physicists measure:| Variable | Symbol | Meaning ||-|-|-|| Particle ID | `id` | A number identifying *what* the particle is (e.g. 11 = electron, 2212 = proton) || Transverse momentum | pT | Momentum perpendicular to the beam (GeV/c) || Pseudorapidity | η (eta) | Describes the *angle* of the particle relative to the beam || Azimuthal angle | φ (phi) | Rotation angle around the beam axis (radians) || Energy | E | Total energy of the particle (GeV) || Mass | m | Rest mass of the particle (GeV/c²) |

In [ ]:
# Simulate one collisionpythia.next()# Collect the final-state particles (the ones that actually reach a detector)part_id = []part_pt = []part_eta = []part_phi = []part_E = []part_m = []for particle in pythia.event:if particle.isFinal(): # only keep particles that don't decay furtherpart_id.append(particle.id())part_pt.append(float('%.3g' % particle.pT()))part_eta.append(float('%.3g' % particle.eta()))part_phi.append(float('%.3g' % particle.phi()))part_E.append(float('%.3g' % particle.e()))part_m.append(float('%.3g' % particle.m()))print(f'This collision produced {len(part_id)} final-state particles.')

### Organizing the data into a tableA **DataFrame** is pandas' version of a spreadsheet: rows are events or particles, columns are measured quantities. This is how physicists organize collision data.

In [ ]:
# Build the DataFrameevent = pd.DataFrame({'ID': part_id,'pT': part_pt,'eta': part_eta,'phi': part_phi,'Energy': part_E,'mass': part_m})# Display the tableevent

Each row is one particle. The `ID` column uses the **PDG numbering scheme**: a universal code for every known particle:| ID | Particle ||-|-|| 11 | Electron (e⁻) || -11 | Positron (e⁺) || 211 | Pion (π⁺) || -211 | Pion (π⁻) || 321 | Kaon (K⁺) || 2212 | Proton || 22 | Photon (γ) || 2112 | Neutron |You can look up any ID at the [Particle Data Group](https://pdg.lbl.gov/).**Question:** Look at the ID column. Can you identify any of the particles produced in your collision?

In [ ]:
# How many of each particle type were produced?print('Particle counts in this event:')print(event['ID'].value_counts())

-## Part 4: Making PlotsTables are useful, but physicists think in **histograms**: plots that show how many times each value occurs. Let's visualize our collision.

In [ ]:
# Plot the energy distribution of all final-state particlesplt.figure(figsize=(7, 4))plt.hist(event['Energy'], bins=20, color='steelblue', edgecolor='black')plt.xlabel('Energy (GeV)', fontsize=13)plt.ylabel('Number of particles', fontsize=13)plt.title('Energy of final-state particles: one e+p collision', fontsize=13)plt.tight_layout()plt.show()

In [ ]:
# Plot the transverse momentum (pT) distribution# pT is especially important: particles from a hard scatter have HIGH pTplt.figure(figsize=(7, 4))plt.hist(event['pT'], bins=20, color='tomato', edgecolor='black')plt.xlabel('Transverse Momentum pT (GeV/c)', fontsize=13)plt.ylabel('Number of particles', fontsize=13)plt.title('Transverse momentum of final-state particles', fontsize=13)plt.tight_layout()plt.show()

In [ ]:
# Where are the particles going? Plot eta (angle from beam) vs phi (rotation around beam)# This is how a real detector 'sees' a collision: called the eta-phi planeplt.figure(figsize=(8, 4))plt.scatter(event['eta'], event['phi'],s=event['pT'] * 30, # bubble size = pT (bigger = more energetic)c=event['Energy'], cmap='plasma', alpha=0.7, edgecolors='black', linewidths=0.5)plt.colorbar(label='Energy (GeV)')plt.xlabel('Pseudorapidity η (beam direction →)', fontsize=13)plt.ylabel('Azimuthal angle φ (rad)', fontsize=13)plt.title('One e+p collision in the η-φ plane (bubble size = pT)', fontsize=12)plt.tight_layout()plt.show()print('Each bubble is one particle. Bigger = higher pT. Brighter = more energy.')

-## Part 5: Many Collisions: Finding the Struck QuarkOne collision tells us about *one* event. But physics is statistical: we need to look at **many** collisions to see patterns.In each e+p collision, the virtual photon strikes one of the quarks inside the proton. The proton contains mostly **up** (id=2) and **down** (id=1) quarks, but also a sea of quark-antiquark pairs and gluons.If we record the identity of the struck quark in many events, we can measure the **quark content** of the proton: this is related to the **parton distribution functions (PDFs)** that are central to all of modern particle physics.Let's run 500 collisions and record which quark gets hit each time:

In [ ]:
n_events = 500struck_quark_ids = []print(f'Running {n_events} collisions...')for i in range(n_events):if not pythia.next():continue # skip rare failed events# Find the struck quark: it has status -23 and is not an electron (id != 11)for particle in pythia.event:if particle.status() == -23 and abs(particle.id()) != 11:struck_quark_ids.append(particle.id())breakprint(f'Done! Recorded {len(struck_quark_ids)} struck quarks.')

In [ ]:
# PDG quark ID labels for the x-axisquark_id_labels = {-5: 'b̄', -4: 'c̄', -3: 's̄', -2: 'ū', -1: 'd̄',1: 'd', 2: 'u', 3: 's', 4: 'c', 5: 'b'}import numpy as npbins = np.arange(-5, 7) - 0.5plt.figure(figsize=(9, 5))counts, edges, patches = plt.hist(struck_quark_ids, bins=bins,color='mediumseagreen', edgecolor='black', rwidth=0.85)# Label the x-axis with quark namestick_positions = np.arange(-5, 6)tick_labels = [quark_id_labels.get(i, str(i)) for i in tick_positions]plt.xticks(tick_positions, tick_labels, fontsize=13)plt.xlabel('Struck quark identity', fontsize=13)plt.ylabel(f'Number of events (out of {len(struck_quark_ids)})', fontsize=13)plt.title(f'Which quark did the photon hit? ({n_events} e+p collisions)', fontsize=13)plt.tight_layout()plt.show()

### What does this tell us?Look at the histogram. The tallest bars are at **u (up quark, id=2)** and **d (down quark, id=1)**: the proton's *valence* quarks. These are the quarks that "define" the proton (2 up + 1 down).But you also see smaller bars for strange (s), charm (c), and their antiparticles. These come from the **sea quarks**: quark-antiquark pairs that pop in and out of existence inside the proton due to quantum fluctuations.**This is what deep inelastic scattering experiments like HERA were built to measure.** By firing electrons at protons, physicists mapped out the quark content of the proton with extraordinary precision.

In [ ]:
# Let's quantify it: what fraction of the time did we hit a u or d quark?ids = np.array(struck_quark_ids)n_total = len(ids)n_u = np.sum(ids == 2)n_d = np.sum(ids == 1)n_sea = n_total - n_u - n_dprint(f'Up quark (u): {n_u:4d} events ({100*n_u/n_total:.1f}%)')print(f'Down quark (d): {n_d:4d} events ({100*n_d/n_total:.1f}%)')print(f'Sea quarks (other): {n_sea:4d} events ({100*n_sea/n_total:.1f}%)')print(f'\nTotal: {n_total} events')

-## Act 1 Checkpoint: DiscussionBefore we move on, let's take a few minutes to discuss:1. **What did the electron actually "see" inside the proton?**The virtual photon probed the electric charge of individual quarks. The more often a quark gets hit, the more "visible" it is at the energies we're probing.2. **Why are up quarks hit more often than down quarks?**The proton has 2 up quarks and 1 down quark: but also, up quarks carry charge +2/3 while down quarks carry charge −1/3. The photon couples more strongly to larger charges. Both effects mean up quarks are hit more.3. **What do you think happens when we do this with TWO protons instead of an electron and a proton?**The quarks from *both* protons can interact. More quarks, more energy, more particles produced. That's Act 2...4. **And what happens when instead of protons, we smash gold nuclei: each containing 79 protons and 118 neutrons?**Hundreds of quarks and gluons interact simultaneously. The energy density becomes so extreme that quarks are no longer confined inside protons: they break free into a new state of matter.**That's the Quark-Gluon Plasma. That's Act 3.**-**️ Continue to Act 2: Two Protons Collide: Real Data from CERN**